# 04 · Post-processing — heatmaps, top-k, case aggregation

**Introduction to Pathology MIL — Notebook 4 of 5**

The attention vector is the interpretability payoff. Here we:
1. scatter attention back to patch coordinates → an **attention heatmap**,
2. pull the **top-k attended tiles** as an evidence panel,
3. **aggregate slides → patient** for the clinical decision.

> ⚠️ Attention *localises* but does not *prove* causation — treat hotspots as a hypothesis
> for the pathologist, not ground truth.

## ⚙️ Colab setup

Run the cell below **first**. It writes the helper modules (`mil_models.py`, `mil_utils.py`) and regenerates any prerequisite data, so this notebook runs **standalone** in Google Colab — just choose *Runtime → Run all*. Colab already includes torch, numpy and matplotlib.

In [ ]:
# === Colab setup — RUN THIS CELL FIRST ===
# Makes the notebook self-contained: writes the helper modules used below.
# Colab already ships torch / numpy / matplotlib, so no pip install is needed.
import os, base64, gzip

_FILES = {
    'mil_models.py': 'H4sIADC+ImoC/91Z3W7byBW+11McKBdLeiXGEpBiYdQF4m22WcDxLmInN4aXGIkjcSpyRp0ZWnbSAL0qFnu5KFD0efomeZJ+M0NSpKx0nR8DRXlhUvNz5vx+55zxcDgclKJIS5XxwiTrW3r/t3/QS77gmss5pxffnxJbLjVfMqu0oYXSZHNOPzKbq0Itb/2Kuaq04clgcJFzw4lpjqFyzeZ2RJqzjM0KPqKDudKaz+0BiXJd8JJLy6xQ0lDJmbSBNmfzXMjliKSyA0aZVuuxkKCyLtjc72l5UFoshWQFfXv69IVboRJ6ds31LXlpqGRrQ4xmbElqMVgzO89pwZmtNDdEz+n9zz/Ty5/env37n9k7IqvIFCLjBKmENSNiMqNok0MRxNbrQsydFDHOAQFpBo6DNdfjQJdZC9YgDF1DQjBYGZ55TnOc6DiBdghPS91aOSJ+YzWj48Bw9Dym+nlEz48oOhtRhqH3f/+12YfBCVSTzgtmDDcxNM6NxUk42NxK8GTF3ElsCEqrrIBNS7biaTuZZswyw20yGML0C61KStNF5ZSSps4wSluIDu0H2wzqIcg0z3s/EimJGZJydzRZVHLuNsM0WPDdYDB4ROMv94DaYUInkKIQkh9576HHMPcNrZXC2JIokooKzrR0RmuH10yzkluuobkvy5I3CP2IcxAPEXTwQmUV3OXIGx2qfgEmx3BZx+bY8eNdGJYauQ9okpwwTJOnJBaC64Qu/JIgp3PQGXwpcXbzVDO+gO2EFDZNIyxajGD0NBPlEd624yb+N7xsOvKOdkTGavwcOsUNR5SLLOOyWfRkMq25do+p4ONRnLTnxO2UIw2rO4rO2d425IaQcPhuSwGMJX5RcPP+xFqrP2MCCjvnf6lcCLHCqe/UKyMK8jQsxiO38CU/fRWFzz8CHVRlo8Nk+iSO+5S3egz0a4qBUj+GGmUiXDdMZ7UuEYHBoy+4NEp3lJKDYst9J2p3nkc+gmvW281vsDlHUN5EkOz4ME6uWVEBkMSiq6rjoEYCKHO3HKqt13u6tRRbqgEeGr62skdvkkoaaJa/4dFh/CFWA7eAlm+3JB9RJQVUUtKwRbchGUWZ2kh4EGdlA25AezBtADcFbZRema2TAObAVVDkoiqKKHqemJyt+eXh1QhWnCSHiN3OGCxxLeb8+HkSPrYMBeTdQdC37x4AXiYJPT1xic0lw1b4sQvFzCe86HtnFw6kLJIRTQ8n30AUpEhMXzMtkM0eBF/+5E542vCzD2ZYKpwTqAXMcpOKiDY/XdAlWSbz6DXlqYjp/S//IiOWpRJZ9CoMXVF8f1jpw8X0ye9GPksjDo9oUSjmRn1A3gtFvL++/kgIuHDi9CGgZmEXBV59JOXzWjH3Ir7ZiyyTDyJKvosovfDLQ8YPXHVgtgnqTRR0FeUxHdTS4TvewZtJjLiVCFtWiDfwR4Oqi5seucBE7SQRg/1qZNkDXyBnqtIlnwkpFFdN2tqNSoYwDE7qI2efb3rv7VRKTVb+ek/q+6wkdyehBbBI73juR7jowyWqGiJ3Yrvxpg7j/3MJ7oM5rvVZx73z19/YP+mnx4jBwfM4geOFpDeiFedr93mhKx53E1Z9dAelN1wsc+dp2H6P/Bj/Vh7cTTtJk04n8QOln2niW5rx+YlPQMuduPkaDmyA6HM+nhcVyn/toohVN6IQDN1PoYwBFTzRadUmqTPf+NCJUCXP0MwU9EyifeJh9/RwOnmYmthJkp6f7AME/z7H8QUfzzQEyv1qJN8sQ+cm6aARtAMNByFUr4XLxbNbj0cWTeLKNWye4kxZq0o34FSWYZnv0VBjRWvDq0yNCzbjReEmlBFWXHOUINK1uPiMQ9kNzKwKC948yZYPp1vMLasCaR7B5btE1C56jRjyvGzthMIooTNlUXBb30YKQ2c/eHqrsavoDK3RtWo155ANRnDGdiSGW7MO3Sab48/Wvi0zaORWSavMLwiXgztxuh8/EZepYa6Zbwa/+QhIbfZiW/PZX9Ay6wCu+f7/geUt4ii0dzMhe8bt0IEaw0+CQI/BMKlFGNg5u9mcbjebwEUIvVNhbHR5l6lp7K8qUtfHIQ6XPNqyedXJH2l7gIuDpqqBNkaBnVRkNx0HgFdehMhsAtFdZdQx2ITeaBuvndkmGpMGJtxz5lunplNoh1eusxQy6jkVAI8eP6ap66vO6A+oXnwr1ck0mFjR72ly1PP2Gu3zRPJN+oZrZaIo7mc2llwLvonGHVoAoLauwvfKFVWrGPYA0nZcFpL2l433r4Ma23VzBpNhsdfTVedIppfcp7TeOvcJdzLRCmWdvV3z4zBWKHepVvdWedNb7Yn07hO2Bi3ci97VhzrSfZ552frMVZRfulfcZODpyjnlbgr+LpmDE5MiFhG4t1GTkmtV3LfQGZHH/wBZfwVCSwdB7vVJJdBn10CfUwT1q6BPKHf29f37yguf+Jqb18iCgxFt3MWV1QzwLpdxDVEhkW9dGbQ8WvjY3R9UiERvEZfrJELEmaIflV0qXqQdJAoYBHtGnlB83+pt2CMzPNoe5Oo650mzShRZuB6PJCvD5dk9kurBwWpTu5Pb5lAYL4TNxuXEQS22n+pfnuHtGqPhu6NdGZrbxSaptceGa73jQGMfbXZT38t9MmVQ6BN2N1RsVopieIdaaAD30PI6ucse5kvHn3unZraHv6aG/K804YcA+NfuNu2Z1kpHi2ElV1JtZP3/gK/eujPffQVJ/gPZBRiT9xgAAA==',
    'mil_utils.py': 'H4sIADC+ImoC/71a247bxhm+11MMFKQh1xItrbtBqoQBNm4SGIidRbxuUggCM0uOpLEokpmhdlc2DOS21y3QJ+iT9E3yJP3+OfAg7dpNYVcXXmk4/+mb/zj0cDgcbGWe7GqZ66jas99+/Qd7vuZKZGwt8koozZalYvVasAter8u8XO3Z0yffsbTcKS1YUdbiqiw3OhoMLlR5LTOhZwPGTtiWb0Si9wVIa5kmGa+5FjUzEpTgudRYZs0GthS83inBTq74Sp8wXTJxLdS+kQCmjKldoZkosnFdjvGH3UjotKtZVt4UeckzWazY5eNvz9lDxtm3Fy8i9lRuZaqNAcGzf/8zC1ktCl3CLm45QhWYuCugoCwLME/LTCh2U+7yjFWqzHapGBlBYFnlvKiBzbDebUs1ZFquCp5HxuAKDERRJ7pW+LaUIks2yxJMyOJc8A1fibHmS8Ee/4XpKpd1DTmB+cau9p4+tNzARBZJWQiYIq55vuO1IH0Nt60s5BZ600GYjW4Ty8uy0pZBWm6rXS2SragVIdB8iMH5ix++fzxiVxz2pLCHp+lO8XQ/wpOLH/BE1ClO9BKotSe0EoWAaXCHXNS6PXo2OWUchzE583ixE4LVGQH2W0Ca6wFtck+sVgxSr4WO2JMCpDyFFMH25c6hrwTwxsrPd/vSzwNzKuQvTNxWpaKTuWo9hk2mbKnKLayrdwDnx+dPgM0QLm9Wk2S5I49LEia3RA0jQGrcQA/cUrHbIio4rK0GH7Fn319+PWNAIF0zqR0ZhOb8lcz3TBYa/m98zVgOb3xojmW5K1LDlty6XvOa9oAfGTKuYKbBz5+UD7ybUm3AE25nJI6XSgj457VUZbGFp+CAwGP8/j7g9rw57aEPSMJ3aLW+Emt+LVguN+IwcMY+cBCO8Lv3rFgmlvcklKBIXODo+LPJCHglmdzGZ9PTEdNCZPFkNGD3fxBJRJ+uhY5PJ6Df8ttm4U+TtxNXpU6Wxm3LIp5EZyOXD+Jp9Gk4M5TkbfT3BwEscf6cUeJj5ZJlMq31iFGE47SZzuE6s0baa59NZDayz8y3nF+JfORzpUZKG2VsicxXPzoNR4j5UmVm9TR8MzDMztkJ1JS1vBYnlhF2FTW8k5TRyCI4RWcDqeUzm8OAZYrfFC6MDEO9lkvy+W/5TmvJCxaQv/vEaAEIP2eFWHESamVqLxTBWYiIXa4pfLRhKG4hHdFDbCihca0RdKTPzKzB/yjYvBVMLpfst7/9a0rhVlP66j6NeqgrlIMYoRspxFe5jeBGfJfXCdYDco7Q7PoIQCzlLbS32xwGOCElDC5DikIfDrpCSjJ0Zhe8TUEGOEakTcZVlhSlAq6BdcUw4rreVyIYunMahgfUD42KuQRwq4hog+ZZaA+RfB1C5gvziwpyRSpB3ZXoBEDY+o9xFJDIog5IN2tZELIvel4bNgRF4g7K2gI6sUISCqYj9igkjKa//fr3U3+a5LFOasOB1NJdtezejlL0edZRqhHSicJeBIZhj/ane4CGv7u4D+9Am5S/4ulmpShZsVrCv0SPr1x6vGI2nR1F/EfWu9twKdOdplixbuIjJTgDQtOzj80T67fhEa8isUQxmUngEhbPUK7Jrl0hl3T8k2iCXDKJpmdheMxCZrcOh3RdomCS9Y7tyJfM+Buea3FM+9Mc1Av2IHaBSo2G97WjzfbJluuNDaJXQpWapGWEcIwSS4He7rK8Y3apDvBF7Rez/4n9oH8OS5QAdjr+s8XcpjuETQ30S6qTKyUzFpAjrhGtW17pPgdTnq3/QWAqZE5/9S8KZ3AI9R6ueKutapm8RgtDe7l17mfhyDDrk7j0a0jgo+kmmN+Cy14vRozfSh1PW/eEDsY5T9jp2ac9LhTsEa8q9LcBFYmgLQXxcnjxuppNHmVvhm1Z6Kwmz19rPHpb0XK5IT4oJfFPvnzE9s87mLSnF7dfHYTKFDtjyKjjX++/W/mu21WncB89RrslbUcyY01rfXF++eTrZ5eArOnNadlA8AE6lfsmgcAiUiT0Q8dnvkNpe4W/SoG+N7DNP8JpxGAPfQlZee27BPgv/IJSZSHMopklaPThXnT0eyugo2PjL10qDBpen2if9TUNhia32T1roVwE+AoEWa/fNEVKIpapIAh00TQ5CANApyB4sgiNnNMsyObD1uGHiJzXQyNtOGN4ZL9idQhQsDRfvAmP2M0PeCzmZvfCR5V0SksTrNSQBU0JHTQR4gKZK8X3wbzhXS0aLdpKTKwWrlibw6VqPbcbkm5NNA/Dto6DEz2GHCT/X3YisKI7GK1URcxIwlxajtKR3NABBF7ZmJiF88li0dCaYrneLZe5CMAn7NXqlyNSu38+tKmfqY3C85fsY++2DYogbixWbHlsZcuInNihTS275blctPocO7xpd0a+5fmvPYo+gWeBun7gCUTbKENlqZUcHniHqQEmHI+V+wCp7Kkd/j5ANjq4CAj2SY0CjbKUYFq/anOPuxNwFwB3XA34bwyTIKadiF2gcIzNhIxUVDK9yQWnvC+qJv9YYS6StI0lu9aUQlTC0O0lhQ730prvCUxP53timMZ3qkwDSO96LbDEII9e7Me1rAuxZy+Q9uED5r7J3BjAH8aYYxVKh+n5N2jeED5ef/BM+C5NdFoiv0WdFhmtM6VvzDaui9hHmFWCMLR9XDBlY7YP3dqg014aUgrRCYMXOw741XdcVzeNlcGw4MWwZYKSLJTPSCtdomWpRmyDUhAPEQorQUud/ZqyRjU3ZJ2EQMYm2t5aOGYmZtGJ5qIIqpA9YNMDuD3xS0b6v/ODccpi66AlyBF2AJ1RC17phuHNWuYCbL8wsvVh6qG+8GVvxRJsSMeWyNyd6GpulhcEK368XBw3nBvqe6d9xDt4zF/OPAsWvDQyHlhZIXvITqNJj5LAMA/72FpQBebXfWABvQdMs9sdD7Xz3YPpgWnuSleFY1+X8AS5olnRusTgwHkCy3hvZpmFdUV4pXXAExbYL94o+ifwz4xb9qKrUkfRdYcfjjuFBXLZ/tDr6soSpLst6bPv1KH+Exs/bT3H/I3nIH9ImzAxyS124fcDUI5gQ8f6ZicNVi4u79hCwsoiRemguSGYo17S1JQuMMpU92yZYktl9gzuDFUQoEZUrwLaZLiFDsZKmThzaYx9GVPiPM58BiCTQQJLEdP5/IEFPn3S715eqYsjiskBxaRPsXy3jEOKd8s40EqLQnePwZ5U0RyDruwxFf550T9JPzjQ1NNAbZK8DZ/AJfx+BQtHnb1w2Wavcd979/oKl6CuOZLAGPDA6Gnjo8PaVT+/1dhWF6EzhYLdFTYyp0NIPOluStaellZogIQUTAdp5wHJtZThh+gx3vraAI3TlRmrtXxFFXtqbzDef0OC9lDdcJUF5rWAHUG1u9uMn5WFaHuSF/ZGhKV072KuLMwdHDezHrsoy5xsecjOv7J/H393/jR5/lXTftRqP7u3TNFmOlZR1boz1fj3Xc31Yg5Zh4FvVA96mpt/Q3elSUzZJYL8a6VKNXsLOR00odK88PGwuPH5qPkcsZMRE1WZrnX8aALpKj4V4z+O2E0WT8X47HBwJzuSGyFX6zqeRI8w5udca7eiDeAQJq4xUsbDtNoNR26Qwu/PIFWoq1L7GyVriXs5Yl5LHC1ERRH5Fx7wLa7ZN2aPMSuqy8DKskiVgCl2dPgut9F5xrc/WgyiCt0J2la6IgyNobmCmUbxJBMp38c3boJFj9Uzi+6Ei7JmZF2Lfn+Ll2tfCAa9h75q2x3uLrGBqWvBlQC8aBdH9hu1mvS+kEsybDyNcEIWYts/rNGHlmrfv80Vnetce7KdiutwIz8I7uwH3WhfCbXd2ddX7QVCf+Qzg6Oh7HdHVKPI3TBg9oc/ctAGJ3oJkJh2n4Z0f3E0XISHh9ptBnoQz9sJfnEnmM0VVbmSdAgJPP0WxoDRW7NGy/eQjyb9v4lMwkgwBqoS6nv2e+9Nce/ww8Pb4aHPBQkxNK8DjFrHPaYTaP486IYe+itDMj/g1QccIWDuQZOV4lmAloT2RHR7bUzHAu3Qtag6rkAZPG5eEvfTR5Mz7gHbuaOffbF9PjRFtoskEOg+YF82Tt8HwK9Cm+7+zzuRQRdEmxm7jjJR83QdhBEyDv2bU+YLB+z3f8ixN7CUTsXGipGUmC4ijGQttkgfbz73Mdn28cf30mZLb0og220G7O+sFHVGyyGzqZi9FtVscpq9scdhZun4NaHwiUHhk8UserTE06tzdBv2QbcBcc+HPdCNNugZfTo+QBuTwcYnvw7Cd2Y+Cwz9F4mkg05L1eu/nAc533D16W7/6vqWKx6WiWm39NurhlWKGLuTpzv3SvevfuzEbguLj4vZcVaDIv+XnNbJTMm7spLpZfpeo32kWfG6XName3Rc6dX1NJyjZmB2I88Njt5KNK8GDlLeFtocXfUYPJsK2T2Z45Zk1N527qk9bX55Dn7j4D/FS2hjNiQAAA==',
}
for _name, _b in _FILES.items():
    with open(_name, 'w') as _f:
        _f.write(gzip.decompress(base64.b64decode(_b)).decode('utf-8'))
print('helper modules written:', list(_FILES))

# regenerate the synthetic feature bags if this notebook is run on its own
import pickle
if not os.path.exists('synthetic_bags.pkl'):
    from mil_utils import make_synthetic_dataset
    _d, _ = make_synthetic_dataset(n_patients=80, in_dim=512, seed=1)
    pickle.dump(_d, open('synthetic_bags.pkl', 'wb'))
    print('generated synthetic_bags.pkl')

# quick-train a model if mil_model.pt is missing (so this notebook stands alone)
if not os.path.exists('mil_model.pt'):
    import torch, numpy as np
    from mil_models import build_model
    from mil_utils import train_one
    _data = pickle.load(open('synthetic_bags.pkl', 'rb'))
    _IN = _data[0]['features'].shape[1]
    _idx = np.arange(len(_data)); np.random.shuffle(_idx); _c = int(0.85 * len(_idx))
    _m = build_model('clam_sb', _IN, 2)
    _m, _ = train_one(_m, _data, _idx[:_c].tolist(), _idx[_c:].tolist(), epochs=20, device='cpu')
    torch.save({'state_dict': _m.state_dict(), 'model': 'clam_sb', 'in_dim': _IN, 'n_classes': 2}, 'mil_model.pt')
    print('trained mil_model.pt')

In [ ]:
import numpy as np, pickle, torch
import matplotlib.pyplot as plt
from matplotlib import cm
from mil_models import build_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load("mil_model.pt", map_location=DEVICE)
model = build_model(ckpt["model"], ckpt["in_dim"], ckpt["n_classes"])
model.load_state_dict(ckpt["state_dict"]); model.to(DEVICE).eval()

with open("synthetic_bags.pkl", "rb") as f:
    data = pickle.load(f)
sample = next(d for d in data if d["label"] == 1)     # a positive slide
feats  = torch.from_numpy(sample["features"]).to(DEVICE)
coords = sample["coords"]; tumor = sample["tumor_mask"]

with torch.no_grad():
    _, attn, _ = model(feats)
attn = attn.cpu().numpy()

## 1 · Attention heatmap

Each patch has a coordinate and an attention weight; scatter them and colour by weight.
On a real slide you would `alpha`-blend this over the level-3 thumbnail.

In [ ]:
a = (attn - attn.min()) / (attn.ptp() + 1e-8)        # normalise to [0,1]
fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
sc = ax[0].scatter(coords[:, 0], -coords[:, 1], c=a, cmap="magma", s=14)
ax[0].set_title("Attention heatmap"); ax[0].axis("equal"); ax[0].axis("off")
plt.colorbar(sc, ax=ax[0], fraction=.046, label="attention")

ax[1].scatter(coords[~tumor, 0], -coords[~tumor, 1], c="lightgray", s=14, label="benign")
ax[1].scatter(coords[tumor, 0],  -coords[tumor, 1],  c="crimson",  s=20, label="tumor (truth)")
ax[1].set_title("Ground-truth tumor patches"); ax[1].axis("equal"); ax[1].axis("off"); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Quantitative check: does attention concentrate on the planted tumor patches?
print(f"mean attention — tumor : {attn[tumor].mean():.2e}")
print(f"mean attention — benign: {attn[~tumor].mean():.2e}")
print(f"ratio (tumor / benign) : {attn[tumor].mean() / attn[~tumor].mean():.1f}×")

## 2 · Top-k evidence panel

Surface the highest-attention tiles so a pathologist can audit the model in seconds.

In [ ]:
k = 8
top = np.argsort(-attn)[:k]
print("top-k patch indices:", top.tolist())
print("their attention   :", np.round(attn[top], 4).tolist())
print(f"of the top-{k} tiles, {int(tumor[top].sum())} are truly tumor patches")
# On real data you'd read these coords from the WSI and show the tile images:
#   tiles = [slide.read_region(tuple(coords[i]), 0, (P, P)) for i in top]

## 3 · Slide → patient aggregation

A patient may have several slides/blocks. Roll slide probabilities up to a patient
decision (max = "any positive slide ⇒ positive"), then threshold at a validation-chosen
operating point.

In [ ]:
from collections import defaultdict
@torch.no_grad()
def slide_prob(d):
    logits, _, _ = model(torch.from_numpy(d["features"]).to(DEVICE))
    return torch.softmax(logits, 1)[0, 1].item()

by_patient = defaultdict(list)
for d in data:
    by_patient[d["patient_id"]].append((d["slide_id"], slide_prob(d), d["label"]))

THRESH = 0.5
print(f"{'patient':8s} {'slide probs':28s} {'case p(max)':>11s} {'call':>8s} truth")
for pid, slides in list(by_patient.items())[:8]:
    probs = [p for _, p, _ in slides]
    case_p = max(probs); truth = max(l for *_, l in slides)
    call = "TUMOR" if case_p >= THRESH else "benign"
    print(f"{pid:8s} {str([round(p,2) for p in probs]):28s} {case_p:11.2f} {call:>8s} {truth}")

### ✅ Takeaways
- The heatmap should concentrate on tumor — quantify it (tumor/benign attention ratio).
- Top-k tiles = a fast human-auditable evidence panel.
- Report at the level the clinic acts on — usually the **patient**.

**Next:** `05 · Evaluation & Visualization` — metrics, ROC curves, and UMAP.